# 🧪 Lekcja 5: Inżynieria Cech (*Feature Engineering*) na Titanicu — wielkość rodziny i tajemnica tytułu `Master`

W kursie *House Prices* przekonaliśmy się, że dodanie piwnicy, parteru i piętra w jedną cechę `TotalSF` dało modelowi ogromny skok jakości. Na Titanicu mamy dokładnie taką samą sytuację!

Spójrz na trzy surowe kolumny z pliku `../data/train.csv`, które na pierwszy rzut oka wydają się mało użyteczne albo porozrzucane:
1. **`SibSp`** (liczba rodzeństwa/małżonków na pokładzie) oraz **`Parch`** (liczba rodziców/dzieci na pokładzie),
2. **`Name`** — pełne imię i nazwisko pasażera, np. `'Braund, Mr. Owen Harris'` albo `'Palsson, Master. Gosta Leonard'`.

Początkujący często wyrzuca kolumnę `Name` do kosza, mówiąc: *„Po co modelowi cudze nazwisko? Przecież każde jest inne!”*.  
Prawdziwy Data Scientist patrzy jednak uważniej i dostrzega, że **w każdym nazwisku przed kropką kryje się tytuł społeczny (`Mr.`, `Mrs.`, `Miss.`, `Master.`, `Dr.`, `Rev.`)**! Zobaczmy, jakie skarby wyciągniemy z rodzin i tytułów!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1. Wczytujemy dane z folderu ../data/
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')

# 2. Tworzymy dwie nowe cechy rodzinne:
#    A) FamilySize = SibSp (rodzeństwo/małżonek) + Parch (rodzice/dzieci) + 1 (sam pasażer!)
#    B) IsAlone = 1 jeśli płynął całkowicie sam (FamilySize == 1), w przeciwnym razie 0
for df in [train, test]:
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

przezycie_rodzina = (train.groupby('FamilySize')['Survived'].mean() * 100).round(1)
przezycie_sam = (train.groupby('IsAlone')['Survived'].mean() * 100).round(1)

print('--- Szansa przeżycia według wielkości rodziny (FamilySize) [%] ---')
print(przezycie_rodzina)

# 3. WIZUALIZACJA: Po lewej wielkość rodziny (1..11 osób), po prawej Samotni vs Z rodziną!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Wykres lewy: FamilySize a przeżycie
rozmiary = [str(r) for r in przezycie_rodzina.index]
kolory_rodzin = ['salmon' if r == 1 or r >= 5 else 'mediumseagreen' for r in przezycie_rodzina.index]
ax1.bar(rozmiary, przezycie_rodzina.values, color=kolory_rodzin, edgecolor='black', width=0.6)
ax1.axhline(38.4, color='red', linestyle='--', linewidth=1.5, label='Średnia statku (38.4%)')
ax1.set_title('1. Wielkość rodziny (FamilySize) a szansa przeżycia')
ax1.set_xlabel('Łączna liczba członków rodziny na pokładzie (FamilySize)')
ax1.set_ylabel('Procent ocalałych (%)')
ax1.set_ylim(0, 85)
for i, v in enumerate(przezycie_rodzina.values):
    ax1.text(i, v + 2, f'{v}%', ha='center', fontweight='bold', fontsize=9.5)
ax1.legend()
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Wykres prawy: IsAlone (Z rodziną vs Samotny)
etykiety_sam = ['0: Płynął z rodziną\n(354 pasażerów)', '1: Płynął samotnie\n(537 pasażerów)']
ax2.bar(etykiety_sam, przezycie_sam.values, color=['mediumseagreen', 'salmon'], edgecolor='black', width=0.45)
ax2.set_title('2. Szansa przeżycia: Z rodziną vs Samotnie (IsAlone)')
ax2.set_ylabel('Procent ocalałych (%)')
ax2.set_ylim(0, 70)
for i, v in enumerate(przezycie_sam.values):
    ax2.text(i, v + 2, f'{v}%', ha='center', fontweight='bold', fontsize=12)
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 💡 Co odkryliśmy na wykresie wielkości rodziny (`FamilySize`)? (Zjawisko „Złotego Środka”)

Spójrz na lewy wykres powyżej — słupki dzielą pasażerów na 3 wyraźne grupy:
1. 🔴 **Samotni podróżnicy (`FamilySize = 1` / `IsAlone = 1`):** Stanowili większość załogi i młodych mężczyzn emigrujących za pracą do Ameryki. Przeżyło zaledwie **`30.4%`** z nich!
2. 🟢 **Małe rodziny od 2 do 4 osób (`FamilySize = 2, 3, 4`):** Tutaj przeżywalność wystrzeliwuje w górę: **`55.3%`** (2 osoby), **`57.8%`** (3 osoby) i aż **`72.4%`** (4 osoby — najczęściej rodzice z dwójką małych dzieci!).
3. 🔴 **Bardzo liczne rodziny od 5 do 11 osób (`FamilySize >= 5`):** Nagle szansa przeżycia gwałtownie spada do **`13.6% – 20.0%`**, a dla rodzin 8- i 11-osobowych wynosi **`0.0%`**! Dlaczego? Ponieważ wielodzietne rodziny emigrantów mieszkały w labiryncie korytarzy 3. klasy i w chaosie ewakuacji rodzice szukali wszystkich swoich dzieci pod pokładem, nie wsiadając do szalup bez nich.

#### Dlaczego we wzorze `df['SibSp'] + df['Parch'] + 1` dodajemy na końcu `+ 1`?
* Jeśli mąż płynie tylko z żoną, to w kolumnie `SibSp` ma wpisane `1`, a w `Parch` `0`. Gdybyśmy nie dodali **`+ 1`** (samego pasażera), wyszłoby nam `1`, choć razem stanowią **2-osobową rodzinę**!
* A jeśli ktoś płynie zupełnie sam (`SibSp = 0`, `Parch = 0`), to po dodaniu `+ 1` jego `FamilySize` wynosi eleganckie **`1`**.

In [ ]:
# 1. Wyciągamy tytuł grzecznościowy z kolumny Name (słowo stojące tuż przed kropką, np. 'Mr.', 'Master.')
rzadkie_tytuly = ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']

for df in [train, test]:
    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    # Łączymy francuskie odpowiedniki: Mlle (Mademoiselle) -> Miss, Mme (Madame) -> Mrs
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    # Rzadkie tytuły oficerskie/arystokratyczne zbieramy w jedną grupę 'Inne'
    df['Title'] = df['Title'].replace(rzadkie_tytuly, 'Inne')

# 2. Zobaczmy statystyki dla każdego z 5 tytułów: liczba osób, mediana wieku i procent ocalałych!
statystyki_tytulow = train.groupby('Title').agg(
    Liczba_Osob=('Survived', 'count'),
    Mediana_Wieku=('Age', 'median'),
    Procent_Ocalalych=('Survived', lambda s: round(s.mean() * 100, 1))
)
print('--- Podsumowanie tytułów wyciągniętych z kolumny Name ---')
print(statystyki_tytulow)

# 3. INTELIGENTNE UZUPEŁNIANIE WIEKU (Age) — osobna mediana wieku dla każdego tytułu!
mediany_wg_tytulu = train.groupby('Title')['Age'].median()
for df in [train, test]:
    df['Age_Smart'] = df['Age'].fillna(df['Title'].map(mediany_wg_tytulu))

# 4. WIZUALIZACJA: Po lewej mediana wieku dla każdego tytułu, po prawej szansa przeżycia wg tytułu!
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

kolejnosc = ['Master', 'Miss', 'Mr', 'Mrs', 'Inne']
etykiety_tyt = ['Master\n(Chłopcy)', 'Miss\n(Panny)', 'Mr\n(Mężczyźni)', 'Mrs\n(Mężatki)', 'Inne\n(Oficerowie/Dr)']
wieki = [statystyki_tytulow.loc[t, 'Mediana_Wieku'] for t in kolejnosc]
szanse = [statystyki_tytulow.loc[t, 'Procent_Ocalalych'] for t in kolejnosc]

# Wykres lewy: Mediana wieku wg tytułu
ax1.bar(etykiety_tyt, wieki, color=['gold', 'plum', 'steelblue', 'orchid', 'lightgray'], edgecolor='black', width=0.5)
ax1.set_title('1. Mediana wieku dla każdego tytułu (Dlaczego Master to skarb?)')
ax1.set_ylabel('Mediana wieku (w latach)')
ax1.set_ylim(0, 58)
for i, v in enumerate(wieki):
    ax1.text(i, v + 1.5, f'{v} lat', ha='center', fontweight='bold')
ax1.grid(axis='y', linestyle=':', alpha=0.6)

# Wykres prawy: Szansa przeżycia wg tytułu
ax2.bar(etykiety_tyt, szanse, color=['gold', 'plum', 'steelblue', 'orchid', 'lightgray'], edgecolor='black', width=0.5)
ax2.axhline(38.4, color='red', linestyle='--', linewidth=1.5, label='Średnia statku (38.4%)')
ax2.set_title('2. Szansa przeżycia według tytułu pasażera (Title)')
ax2.set_ylabel('Procent ocalałych (%)')
ax2.set_ylim(0, 95)
for i, v in enumerate(szanse):
    ax2.text(i, v + 2, f'{v}%', ha='center', fontweight='bold')
ax2.legend()
ax2.grid(axis='y', linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

### 🔬 Dlaczego wyciągnięcie `Title` z nazwiska to podwójny strzał w dziesiątkę?

#### 1. Odkrycie tytułu `Master` (Mały chłopiec!)
Spójrz na oba wykresy powyżej:
* Gdyby model patrzył tylko na płeć (`Sex = 'male'`), wrzuciłby wszystkich chłopców do jednego worka z dorosłymi mężczyznami (gdzie szansa przeżycia to marne `15.7%`).
* Ale w 1912 roku w języku angielskim każdy nieletni chłopiec miał przed imieniem wpisany tytuł **`Master.`**! Spójrz na tabelkę: na pokładzie było **40 chłopców z tytułem `Master`**, ich mediana wieku to zaledwie **`3.5 roku`**, a przeżyło z nich aż **`57.5%`** (prawie 4 razy częściej niż dorośli panowie `Mr` z wynikiem `15.7%`!).

#### 2. Genialne uzupełnianie brakującego wieku po tytule (`Age_Smart`)
W wśród 40 chłopców z tytułem `Master` aż 4 nie miało wpisanego wieku (`Age = NaN`).
* W Lekcji 3 (gdy używaliśmy jednej ogólnej mediany dla całego statku) wpisaliśmy tym czterem małym chłopcom wiek **`28.0` lat** — robiąc z przedszkolaków dorosłych mężczyzn!
* Teraz, używając `df['Age'].fillna(df['Title'].map(mediany_wg_tytulu))`, komputer patrzy na tytuł pasażera:
  * Brakuje wieku u **`Master`**? Wstawia **`3.5` roku**!
  * Brakuje wieku u panny (**`Miss`**)? Wstawia **`21.0` lat**!
  * Brakuje wieku u mężatki (**`Mrs`**)? Wstawia **`35.0` lat**!

#### 3. Jak działa wzorzec `df['Name'].str.extract(r' ([A-Za-z]+)\.')`?
To tzw. **wyrażenie regularne (*Regex*)** — sprytny szablon do wycinania tekstu:
* Spacja na początku `' '` mówi: *„zacznij szukać po spacji”*,
* `([A-Za-z]+)` mówi: *„wytnij ciąg liter od A do Z (wielkich lub małych)”*,
* `\.` na końcu mówi: *„który kończy się kropką `.`”* (dzięki temu z napisu `'Palsson, Master. Gosta'` wycina idealnie słowo `'Master'`).

In [ ]:
# 🎯 ĆWICZENIE 1: Sprawdźmy, jakich pasażerów kryje kategoria 'Inne' w kolumnie Title!
#
# Do kategorii 'Inne' trafiły rzadkie tytuły kapitanów, pułkowników, lekarzy, pastorów i arystokracji.
# 1. Wyfiltruj z tabeli `train` wiersze, gdzie train['Title'] == 'Inne'
# 2. Wyświetl dla nich kolumny: ['Name', 'Pclass', 'Sex', 'Age_Smart', 'Survived']
#    Kto z tej elitarnej grupy przeżył, a kto został na tonącym statku do końca?

# Miejsce na Twój kod:


In [ ]:
# 🎯 ĆWICZENIE 2: Porównajmy wiek uzupełniony starą metodą (28 lat) z nową metodą (Age_Smart)
# dla chłopców z tytułem 'Master', którym brakowało wieku!
#
# Wyfiltruj z tabeli `train` wiersze, gdzie:
# (train['Title'] == 'Master') & (train['Age'].isnull())
# i wyświetl dla nich kolumny: ['Name', 'Title', 'Age', 'Age_Smart', 'Survived']

# Miejsce na Twój kod:


---
### 📌 Podsumowanie lekcji w 3 punktach

1. **Cechy rodzinne (`FamilySize` i `IsAlone`):** Połączenie `SibSp + Parch + 1` ujawnia, że największe szanse ratunku miały małe rodziny 2–4 osobowe (`55–72%`), podczas gdy osoby samotne (`30.4%`) oraz bardzo liczne rodziny z 3. klasy (`0–20%`) ginęły znacznie częściej.
2. **Wyciąganie tytułu (`Title`) z kolumny `Name`:** Za pomocą `.str.extract(r' ([A-Za-z]+)\.')` wydobyliśmy tytuł społeczny pasażera, odkrywając m.in. grupę 40 małych chłopców oznaczonych tytułem **`Master`** (przeżywalność `57.5%` wobec `15.7%` dla dorosłych mężczyzn `Mr`).
3. **Inteligentna imputacja wieku po tytule:** Zamiast wstawiać każdemu pasażerowi bez wieku ogólną medianę `28.0` lat, przypisujemy mu medianę wieku właściwą dla jego tytułu (`3.5` roku dla `Master`, `21.0` lat dla `Miss`, `30.0` lat dla `Mr`, `35.0` lat dla `Mrs`).